In [1]:
import os
import numpy as np
import pandas as pd

import csv
import sys
sys.path.append("../")

from joblib import dump
from joblib import load

from VRAI2.main_train_xgb import main
from VRAI2.utils.preprocessing import group_wavelengths
from VRAI2.utils.data import y_columns, get_x_y_labels, col_names_switch
from VRAI2.utils.models import train_xgb_regr, compute_regression_metrics, train_mlp_regr

In [2]:
file_path = "../../../data/raw/_datasets_pretraining/"
file_name = "CornSilage_Rev483.xlsx"
sheet_name = "DATASET_new"
beams_step = 1

out_path = os.path.join("./", "outputs", "xgb_new_data", file_name[:file_name.find("_")])
os.makedirs(out_path, exist_ok=True)

# Load data

## New data

In [3]:
with pd.ExcelFile(os.path.join(file_path, file_name)) as xls:
    data = pd.read_excel(xls, sheet_name="DATASET")
    data.columns = data.columns.map(str)
    train_data = data.sample(frac=0.7, random_state=42)
    val_data = data.drop(train_data.index)

    X_train_new, X_val_new, y_train_new, y_val_new = {}, {}, {}, {}
    col_names = y_columns[sheet_name]

    print("\tTarget columns:", col_names)
    for col_idx, y_col in enumerate(col_names):
        # print("\t\tGetting values for target:", y_col)

        if y_col not in y_columns[sheet_name]:
            print(f"{y_col} not in {y_columns[sheet_name]}")
            y_col_act = col_names_switch[y_col]
            print(f"using y_col {y_col_act}")
        else:
            y_col_act = y_col
        
        # Training set
        print(col_idx, y_col)
        X_train_new[y_col_act], y_train_new[y_col_act] = get_x_y_labels(train_data, y_col)

        # Validation set
        X_val_new[y_col_act], y_val_new[y_col_act] = get_x_y_labels(val_data, y_col)
        # remove all rows with 0 in y_train and y_val
        mask_train = y_train_new[y_col_act] != 0
        mask_val = y_val_new[y_col_act] != 0

        # Count non-zero values in each target column
        # print(f"\t\t\tNon-zero values in training set for {y_col_act}: {len(y_train[y_col_act])}/{len(train_data)}")
        # print(f"\t\t\tNon-zero values in validation set for {y_col_act}: {len(y_val[y_col_act])}/{len(val_data)}")
        
        # Beams grouping
        X_train_new[y_col_act] = group_wavelengths(X_train_new[y_col_act], beams_step)
        X_val_new[y_col_act] = group_wavelengths(X_val_new[y_col_act], beams_step)

        # Apply masks to filter out zero values. Only if non-zero values > half of total
        if len(y_train_new[y_col_act]) > len(train_data) / 2:
            X_train_new[y_col_act] = X_train_new[y_col_act][mask_train]
            y_train_new[y_col_act] = y_train_new[y_col_act][mask_train]
            X_val_new[y_col_act] = X_val_new[y_col_act][mask_val]
            y_val_new[y_col_act] = y_val_new[y_col_act][mask_val]

	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS']
0 SS
1 AMIDO_SS
2 PG_SS
3 ADF_SS
4 NDF_SS
5 CEN_SS
6 EE_SS


## Old data

In [4]:
in_path = "../../../data/raw/Grain"
sheet_names = ["DATASET"]

X_train_old, X_val_old, y_train_old, y_val_old = {}, {}, {}, {}
for dir in os.listdir(in_path):
    dataset_name = dir.split('_')[1][:-len('Grain')]
    for file in os.listdir(os.path.join(in_path, dir)):
        if file.endswith("DATASET.xlsx"):
            print(f"\nProcessing dataset: {dataset_name}, file: {file}")
            X_train_old[dataset_name], X_val_old[dataset_name], y_train_old[dataset_name], y_val_old[dataset_name] = main(
                dir, sheet_names, os.path.join(in_path, dir, file), beams_step)


Processing dataset: Corn, file: 08-MG_Rev489_DATASET.xlsx
	Target columns: ['DM', 'Starch', 'Protein', 'ADF', 'NDF', 'Ash', 'Crude Fat', 'Crude Fib.']

Processing dataset: Wheat, file: 09-FG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Rapeseed, file: 143-RG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Barley, file: 25-OR_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']

Processing dataset: Soybean, file: 59-SG_Rev489_DATASET.xlsx
	Target columns: ['SS', 'AMIDO_SS', 'PG_SS', 'ADF_SS', 'NDF_SS', 'CEN_SS', 'EE_SS', 'FG_SS']


# Models

## Train on new data (optional)

In [60]:
# for y_col in y_train:
#     # Train regression and classification models
#     print(f"\nTraining combined SVM model for target: {y_col}")
#     results = train_xgb_regr(X_train[y_col], X_val[y_col], y_train[y_col], y_val[y_col], 
#                      y_col, out_path, beams_step)
    
#     model_out_path = os.path.join(out_path, f"{y_col}_regr.joblib")
#     dump(results["model"], model_out_path)

#     with open(os.path.join(out_path, f"{y_col}_regr_selected_freq.csv"), 'w') as fil:
#         wr = csv.writer(fil)
#         wr.writerow(results["selected_indices"])

## Load new models

In [5]:
model_path = out_path

# Carica il modello
models_new = {}
for model in os.listdir(model_path):
    if model.endswith(".joblib"):
        model_name = model.split("_")[0]
        model_full_path = os.path.join(model_path, model)
        print(f"Loading model: {model_name} from {model_full_path}")
        loaded_model = load(model_full_path)
        models_new[model_name] = loaded_model

Loading model: ADF from ./outputs/xgb_new_data/CornSilage/ADF_SS_mlp_regr.joblib
Loading model: ADF from ./outputs/xgb_new_data/CornSilage/ADF_SS_regr.joblib


/mnt/c/Users/aless/Desktop/ricerca/deep-nir/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MLPRegressor from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/c/Users/aless/Desktop/ricerca/deep-nir/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/c/Users/aless/Desktop/ricerca/deep-nir/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to 

Loading model: AMIDO from ./outputs/xgb_new_data/CornSilage/AMIDO_SS_mlp_regr.joblib
Loading model: AMIDO from ./outputs/xgb_new_data/CornSilage/AMIDO_SS_regr.joblib
Loading model: CEN from ./outputs/xgb_new_data/CornSilage/CEN_SS_mlp_regr.joblib
Loading model: CEN from ./outputs/xgb_new_data/CornSilage/CEN_SS_regr.joblib
Loading model: EE from ./outputs/xgb_new_data/CornSilage/EE_SS_mlp_regr.joblib
Loading model: EE from ./outputs/xgb_new_data/CornSilage/EE_SS_regr.joblib
Loading model: NDF from ./outputs/xgb_new_data/CornSilage/NDF_SS_mlp_regr.joblib
Loading model: NDF from ./outputs/xgb_new_data/CornSilage/NDF_SS_regr.joblib
Loading model: PG from ./outputs/xgb_new_data/CornSilage/PG_SS_mlp_regr.joblib
Loading model: PG from ./outputs/xgb_new_data/CornSilage/PG_SS_regr.joblib
Loading model: SS from ./outputs/xgb_new_data/CornSilage/SS_mlp_regr.joblib
Loading model: SS from ./outputs/xgb_new_data/CornSilage/SS_regr.joblib


## Load old models

In [62]:
model_path = "./outputs/xgb_trained"

# Carica il modello
models_old = {}
for model in os.listdir(model_path):
    if model.endswith(".joblib"):
        model_name = model.split("_")[0]
        model_full_path = os.path.join(model_path, model)
        print(f"Loading model: {model_name} from {model_full_path}")
        loaded_model = load(model_full_path)
        models_old[model_name] = loaded_model

Loading model: Protein from ./outputs/xgb_trained/Protein_regr.joblib
Loading model: Ash from ./outputs/xgb_trained/Ash_regr.joblib
Loading model: Crude Fib. from ./outputs/xgb_trained/Crude Fib._regr.joblib
Loading model: Crude Fat from ./outputs/xgb_trained/Crude Fat_regr.joblib
Loading model: DM from ./outputs/xgb_trained/DM_regr.joblib
Loading model: Starch from ./outputs/xgb_trained/Starch_regr.joblib
Loading model: class.joblib from ./outputs/xgb_trained/class.joblib
Loading model: ADF from ./outputs/xgb_trained/ADF_regr.joblib
Loading model: NDF from ./outputs/xgb_trained/NDF_regr.joblib


# Test old models on new data

In [84]:
train_dict_ON = {}

for model_name, model in models_old.items():
    if not "class" in model_name:
        y_col = [key for key, val in col_names_switch.items() if val==model_name][0]
        if not "FG" in y_col: 
            print(f"Testing {model_name} on the new data")
            train_dict_ON[y_col] = {}
            
            if isinstance(X_val_new[y_col], pd.DataFrame):
                X_val = X_val_new[y_col].to_numpy()
            if isinstance(y_val_new[y_col], pd.Series):
                y_val = y_val_new[y_col].to_numpy()
            
            preds = model.predict(X_val)
            y_val_array = y_val.ravel()

            rmse = np.sqrt(((preds - y_val_array) ** 2).mean())
            print(f"Validation RMSE: {rmse}")

            metrics = compute_regression_metrics(y_val_array, preds)
            for key, value in metrics.items():
                train_dict_ON[y_col][key] = value

Testing Protein on the new data
Validation RMSE: 10.453299129413095
Testing Ash on the new data
Validation RMSE: 1.5725860742206028
Testing Crude Fat on the new data
Validation RMSE: 2.0038102205746227
Testing DM on the new data
Validation RMSE: 31.853006318706278
Testing Starch on the new data
Validation RMSE: 34.01505194425753
Testing ADF on the new data
Validation RMSE: 20.284977755013077
Testing NDF on the new data
Validation RMSE: 29.671957851871564


# Test new models on old data

In [83]:
train_dict_NO = {}

for key, val in col_names_switch.items():
    y_col = col_names_switch[key]
    train_dict_NO[y_col] = {}
    print(f"Testing {y_col} on the new data")
    
    if isinstance(X_val_old, pd.DataFrame):
        X_val = X_val_old.to_numpy()[y_col]
    if isinstance(y_val_old, pd.Series):
        y_val = y_val_old[y_col].to_numpy()
    
    if not "FG" in key:
        if key != "SS":
            preds = models_new[key[:key.find("_SS")]].predict(X_val)
        else:
            preds = models_new[key].predict(X_val)
        y_val_array = y_val.ravel()

        rmse = np.sqrt(((preds - y_val_array) ** 2).mean())
        print(f"Validation RMSE: {rmse}")

        metrics = compute_regression_metrics(y_val_array, preds)
        for key, value in metrics.items():
            train_dict_NO[y_col][key] = value

Testing DM on the new data
Validation RMSE: 11.136560047501419
Testing Starch on the new data
Validation RMSE: 14.600577319016407
Testing Protein on the new data
Validation RMSE: 35.07187529073876
Testing ADF on the new data
Validation RMSE: 18.973200424888734
Testing NDF on the new data
Validation RMSE: 1.67493972828859
Testing Ash on the new data
Validation RMSE: 38.2780430306977
Testing Crude Fat on the new data
Validation RMSE: 39.35088835238084
Testing Crude Fib. on the new data


# Fine-tune a neural network

In [6]:
for y_col in y_train_new:
    # Train regression and classification models
    print(f"\nTraining combined MLP model for target: {y_col}")
    results = train_mlp_regr(y_train_new[y_col], X_val_new[y_col], y_train_new[y_col], y_val_new[y_col], 
                     y_col, out_path, beams_step)
    
    model_out_path = os.path.join(out_path, f"{y_col}_MLP_regr.joblib")
    dump(results["model"], model_out_path)

    with open(os.path.join(out_path, f"{y_col}_MLP_regr_selected_freq.csv"), 'w') as fil:
        wr = csv.writer(fil)
        wr.writerow(results["selected_indices"])


Training combined MLP model for target: SS


ValueError: 
All the 180 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/alecacciatore/rolex/deep-nir/src/deep_nir/VRAI/venv_nir/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/alecacciatore/rolex/deep-nir/src/deep_nir/VRAI/venv_nir/lib/python3.11/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/alecacciatore/rolex/deep-nir/src/deep_nir/VRAI/venv_nir/lib/python3.11/site-packages/sklearn/pipeline.py", line 613, in fit
    Xt = self._fit(X, y, routed_params, raw_params